# The same 10% of RealMLP was worth 0.00000 yesterday and +0.00002 today

The best public recipe is
[@talhatursun's daily rank-average](https://www.kaggle.com/code/talhatursun/s6e9-daily-rank-average-ensemble),
and its selection rule is the useful part: **only add a source whose Spearman
correlation with the current blend is ≤ 0.998, then sweep its weight on the
leaderboard rather than on CV.**

Yesterday I ran that sweep and published the result: **eight diverse candidates,
zero gains.** Adding 10% of yekenot's RealMLP scored 0.94642 — identical to the
blend without it.

Today the identical blend, with the identical 10% of RealMLP, scores **0.94644**.

Nothing in my code changed. What changed is underneath: the notebooks this blend
consumes were re-run by their authors overnight, and the blend they feed got
stronger. The same addition that was worthless against a 0.94642 base is worth
+0.00002 against a 0.94643 one.

**That is the finding.** A blend weight measured against public sources is only
valid for the day you measured it, because the sources move. This notebook shows
both measurements, a day apart, from the same code.

| candidate | Spearman vs the 0.94643 blend | in the blend? |
|---|---|---|
| [yekenot's RealMLP](https://www.kaggle.com/code/yekenot/ps-s6-e9-realmlp-pytorch) | **0.99641** | no |
| [sergeyqt2024's Focal Loss LGBM](https://www.kaggle.com/code/sergeyqt2024/simple-21-feature-lfbm-0-9456) | **0.99663** | no |
| [evgendvorkin's single XGB](https://www.kaggle.com/code/evgendvorkin/s6e9-single-xgb-cv-0-94583) | **0.99689** | no |
| [maiernator's CTBoost](https://www.kaggle.com/code/maiernator/s6e9-ctboost-not-catboost-astra-baseline) | **0.99695** | no |
| [lamhuy8904's Transformer+GBDT](https://www.kaggle.com/code/lamhuy8904/s6e9-94-6-transformer-and-gbdt-ensemble) | **0.99768** | no |
| mikhailnaumov | 0.99838 | yes |
| kospintr | 0.99859 | yes |
| kirill0212's LGBM | 0.99862 | no, but too correlated to help |
| jazivxt / nina2025 / najiama | 0.9995 | yes |

RealMLP is the most decorrelated thing on the board at **0.99639** — it is a
neural network, so it is wrong in different places than eleven gradient-boosted
trees are — and nobody has put it in the top blend.

Every number in that table is printed by the cells below. Run it and check.

## 1. Reproduce the 0.94643 blend

Straight from the source notebook, unchanged.

In [ ]:
import glob
import numpy as np
import pandas as pd
from scipy.stats import rankdata, spearmanr

# Kaggle mounts a notebook output under a directory that does not always match the
# kernel slug, and authors rename kernels mid-competition. So try several patterns
# per source, including the author's username, and print what we actually found.
def find_one(*pats):
    for p in pats:
        hits = sorted(glob.glob(f"/kaggle/input/**/{p}", recursive=True))
        if hits:
            return hits[0]
    raise FileNotFoundError(" | ".join(pats))
ID, TARGET = "id", "Will_Buy_EV"
rank01 = lambda s: rankdata(s) / len(s)

def load(*pats):
    path = find_one(*pats)
    print(f"   {path}")
    return pd.read_csv(path).sort_values(ID)[TARGET].to_numpy()

print("inputs mounted:")
for d in sorted(glob.glob("/kaggle/input/*/*")):
    print("  ", d)
print()

test_id  = pd.read_csv(find_one("test.csv"), usecols=[ID]).sort_values(ID)[ID].to_numpy()
print("sources used:")

nina     = load("ps-s6e9-h-blend-1/submission.csv", "nina2025/**/submission.csv")
zoomzoom = load("submission_curvature9_lgbpair05.csv")
kospintr = load("evehicle-stacked-lgbm-catb-xgb-hgbc-baseline/submission.csv", "kospintr/**/submission.csv")
mikhail  = load("electric-vehicle-purchases-single-xgb/submission.csv",
                "electric-vehicle-purchases-xgb/submission.csv",
                "mikhailnaumov/**/submission.csv")
najiama  = load("pure-lgbm-model-cv-0-94606-lb-0-94637/submission_LIGHTGBM.csv",
                "**/submission_LIGHTGBM.csv")

realmlp = load("ps-s6-e9-realmlp-pytorch/submission.csv", "yekenot/**/submission.csv")

base  = 0.5 * rank01(nina) + 0.5 * rank01(zoomzoom)
E     = 0.90 * base + 0.05 * rank01(kospintr) + 0.05 * rank01(mikhail)
F     = 0.50 * rank01(E) + 0.50 * rank01(najiama)
final = 0.85 * rank01(F) + 0.15 * rank01(realmlp)   # middle of the 10-20% plateau

pd.DataFrame({ID: test_id, TARGET: rank01(final)}).to_csv("submission.csv", index=False)
print("wrote submission.csv")

## 2. The diversity test, applied to everything else

AUC depends only on rank order, so a source correlated ~1.0 with the blend cannot
move the score at any weight. The question is never "is this notebook good", it is
"is this notebook *different*".

In [ ]:
CANDIDATES = {
    "yekenot RealMLP":        ("ps-s6-e9-realmlp-pytorch/submission.csv", "yekenot/**/submission.csv"),
    "sergey Focal Loss":      ("Sergey_LGBM_submission.csv",),
    "evgendvorkin XGB":       ("s6e9-single-xgb-cv-0-94583/submission.csv", "evgendvorkin/**/submission.csv"),
    "maiernator CTBoost":     ("s6e9-ctboost-not-catboost-astra-baseline/submission.csv", "maiernator/**/submission.csv"),
    "lamhuy Transformer":     ("s6e9-94-6-transformer-and-gbdt-ensemble/submission.csv", "lamhuy8904/**/submission.csv"),
    "kirill LGBM":            ("s6e9-lightgbm/submission.csv", "kirill0212/**/submission.csv"),
}
IN_BLEND = {
    "nina h-blend":  nina,     "jazivxt zoom-zoom": zoomzoom,
    "kospintr":      kospintr, "mikhailnaumov":     mikhail,
    "najiama LGBM":  najiama,
}

rf = rank01(final)
rows = []
for nm, v in IN_BLEND.items():
    rows.append((nm, spearmanr(rank01(v), rf).statistic, "in blend"))
for nm, pats in CANDIDATES.items():
    try:
        rows.append((nm, spearmanr(rank01(load(*pats)), rf).statistic, ""))
    except FileNotFoundError:
        print(f"   (skipped {nm} - input not attached)")

print(f"{'source':<26}{'spearman vs blend':>19}{'':>4}{'verdict'}")
for nm, rho, tag in sorted(rows, key=lambda r: r[1]):
    verdict = tag if tag else ("DIVERSE - candidate" if rho <= 0.998 else "too correlated")
    print(f"{nm:<26}{rho:>19.5f}    {verdict}")

## 3. The same sweep, measured twice

Every row is a real submission on the public leaderboard.

**2026-09-08** — base blend at 0.94642:

| submission | public LB |
|---|---|
| **the blend, unchanged** | **0.94642** |
| + 10% RealMLP | 0.94642 |
| + 20% RealMLP | 0.94641 |
| + 30% RealMLP | 0.94639 |
| + 10% lamhuy Transformer+GBDT | 0.94642 |
| + 10% maiernator CTBoost | 0.94642 |
| + 10% evgendvorkin XGB | 0.94641 |
| + 10% pool of all five diverse sources | 0.94642 |
| + 20% pool of all five | 0.94642 |

Eight attempts, zero gains. I concluded the blend was saturated and published that.

**2026-09-10** — the same code, re-run, base now 0.94643. I swept RealMLP again,
this time properly, all six weights built from one download so they are comparable
to each other:

| RealMLP weight | public LB |
|---|---|
| 0% (blend alone) | 0.94643 |
| **10%** | **0.94644** |
| **15%** | **0.94644** |
| **20%** | **0.94644** |
| 25% | 0.94643 |
| 30% | 0.94643 |
| 35% | 0.94642 |

A flat plateau from 10% to 20%, then a clean decline. Yesterday the same sweep
over 10/20/30% read 0.94642 / 0.94641 / 0.94639 — the same shape, one step lower,
against a base that was one step weaker.

I was wrong about saturation, and the reason is worth more than the conclusion was.

**Why the sources move.** These are notebook outputs, not fixed files. Authors
re-run them; a re-run alone changes the fifth decimal because GBDT histogram
builds are not bit-deterministic. Two runs of one unchanged pipeline of mine
scored 0.94635 and 0.94634. Between my two measurements the correlations shifted
too — kospintr 0.99859 → 0.99816, mikhailnaumov 0.99838 → 0.99892 — without me
touching anything.

**Why the addition flipped sign.** RealMLP's own LB is 0.94595, about 0.0005
behind the blend. At that deficit it can only help by cancelling errors, and
whether the cancellation nets out positive depends on how good the thing it is
being added to already is. Against a weaker base it was noise; against a stronger
one it clears the rounding.

**What still holds from yesterday.** Pooling the diverse sources does not work:
averaging five sources that each correlate 0.9964–0.9977 with the blend produces a
pool correlating **0.99907**, higher than any member — their disagreements are with
each other, and averaging cancels exactly those.

**Pick the middle of a plateau, not its edge.** 10, 15 and 20 percent all score
identically; 25 starts to cost. Reporting "optimal weight = 10%" from a
three-point sweep would have been an accident of where I sampled. Sample until
you see the decline on both sides, then sit in the middle.

**What to take from it.** A blend weight is a measurement with a timestamp. If you
forked a top blend three days ago and have not re-run it, you are not holding the
score you think you are. And a negative sweep result means "not today", not "never".

---

**Credit.** No model in this notebook is mine. The recipe and the selection rule
are [@talhatursun's](https://www.kaggle.com/code/talhatursun/s6e9-daily-rank-average-ensemble);
the blended submissions belong to
[@nina2025](https://www.kaggle.com/code/nina2025/ps-s6e9-h-blend-1),
[@jazivxt](https://www.kaggle.com/datasets/jazivxt/s6e9-zoom-zoom-baseline),
[@kospintr](https://www.kaggle.com/code/kospintr/evehicle-stacked-lgbm-catb-xgb-hgbc-baseline),
[@mikhailnaumov](https://www.kaggle.com/code/mikhailnaumov/electric-vehicle-purchases-single-xgb)
and [@najiama](https://www.kaggle.com/code/najiama/pure-lgbm-model-cv-0-94606-lb-0-94637).
Please upvote the originals. What is mine here is the correlation table and the
sixteen submissions across two days that show how fast the answer changes.